In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
data = pd.read_csv("churn_training_data.csv")
data

,account_id,active_days,active_users,features_used,engagement_score,churned
0,A1,2,2,2,80,0
1,A2,2,2,2,85,0
2,A3,1,1,2,45,1
3,A4,1,1,1,30,1
4,A5,3,3,3,90,0
5,A6,0,0,1,10,1


In [8]:
X = data[
    ["active_days", "active_users", "features_used", "engagement_score"]
]

y = data["churned"]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [15]:
model = LogisticRegression()
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [16]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [17]:
new_account = pd.DataFrame({
    "active_days": [1],
    "active_users": [1],
    "features_used": [1],
    "engagement_score": [35]
})

prediction = model.predict(new_account)
prediction

array([1])

In [13]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values(by="coefficient", ascending=False)

feature_importance

,feature,coefficient
2,features_used,-0.005191
0,active_days,-0.010308
1,active_users,-0.010308
3,engagement_score,-0.233511


In [18]:
y_prob = model.predict_proba(X_test)

y_prob

array([[1.09310702e-01, 8.90689298e-01],
       [9.99652787e-01, 3.47213433e-04]])

In [20]:
churn_prob = y_prob[:, 1]
churn_prob

array([8.90689298e-01, 3.47213433e-04])

In [21]:
def assign_risk_band(prob):
    if prob < 0.3:
        return "Low"
    elif prob < 0.6:
        return "Medium"
    else:
        return "High"

risk_bands = [assign_risk_band(p) for p in churn_prob]
risk_bands

['High', 'Low']

In [22]:
results = X_test.copy()
results["Actual_Churn"] = y_test.values
results["Churn_Probability"] = churn_prob
results["Risk_Band"] = risk_bands

results

,active_days,active_users,features_used,engagement_score,Actual_Churn,Churn_Probability,Risk_Band
2,1,1,2,45,1,0.890689,High
4,3,3,3,90,0,0.000347,Low


In [23]:
new_account = pd.DataFrame({
    "active_days": [1],
    "active_users": [1],
    "features_used": [1],
    "engagement_score": [35]
})

prob = model.predict_proba(new_account)[0][1]
risk = assign_risk_band(prob)

prob, risk

(np.float64(0.9870490653433336), 'High')

In [24]:
results["Churn_Probability"] = results["Churn_Probability"].round(3)
results

,active_days,active_users,features_used,engagement_score,Actual_Churn,Churn_Probability,Risk_Band
2,1,1,2,45,1,0.891,High
4,3,3,3,90,0,0.000,Low


In [25]:
def alert_level(prob):
    if prob > 0.8:
        return "Hard Alert"
    elif prob > 0.6:
        return "Soft Alert"
    elif prob > 0.3:
        return "Monitor"
    else:
        return "No Action"

In [26]:
results["Alert_Level"] = results["Churn_Probability"].apply(alert_level)
results

,active_days,active_users,features_used,engagement_score,Actual_Churn,Churn_Probability,Risk_Band,Alert_Level
2,1,1,2,45,1,0.891,High,Hard Alert
4,3,3,3,90,0,0.000,Low,No Action


In [27]:
time_data = pd.read_csv("churn_time_series.csv")

time_data["engagement_delta"] = (
    time_data["engagement_curr"] - time_data["engagement_prev"]
)

time_data["active_days_delta"] = (
    time_data["active_days_curr"] - time_data["active_days_prev"]
)

time_data

,account_id,engagement_prev,engagement_curr,active_days_prev,active_days_curr,churned,engagement_delta,active_days_delta
0,A1,85,80,5,5,0,-5,0
1,A2,90,60,5,3,1,-30,-2
2,A3,70,65,4,4,0,-5,0
3,A4,60,30,4,1,1,-30,-3
4,A5,88,85,5,5,0,-3,0
5,A6,50,20,3,1,1,-30,-2


In [28]:
X_time = time_data[
    ["engagement_prev", "engagement_curr",
     "engagement_delta",
     "active_days_delta"]
]

y_time = time_data["churned"]

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X_time,
    y_time,
    test_size=0.3,
    random_state=42,
    stratify=y_time
)

model_time = LogisticRegression()
model_time.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [30]:
pd.DataFrame({
    "feature": X_time.columns,
    "coefficient": model_time.coef_[0]
}).sort_values(by="coefficient", ascending=False)

,feature,coefficient
3,active_days_delta,-0.015541
0,engagement_prev,-0.058635
2,engagement_delta,-0.131749
1,engagement_curr,-0.190384


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [4]:
from lifelines import CoxPHFitter

In [5]:
survival_data = pd.read_csv("churn_survival.csv")
survival_data

,account_id,tenure_days,engagement_score,usage_trend,churned
0,A1,400,85,0.05,0
1,A2,180,40,-0.30,1
2,A3,300,70,-0.05,0
3,A4,120,30,-0.40,1
4,A5,500,90,0.10,0
5,A6,90,20,-0.50,1


In [6]:
cph = CoxPHFitter()

In [8]:
survival_data.head()

,account_id,tenure_days,engagement_score,usage_trend,churned
0,A1,400,85,0.05,0
1,A2,180,40,-0.30,1
2,A3,300,70,-0.05,0
3,A4,120,30,-0.40,1
4,A5,500,90,0.10,0


In [9]:
survival_model_data = survival_data.drop(columns=["account_id"])

In [10]:
from lifelines import CoxPHFitter

cph = CoxPHFitter()

cph.fit(
    survival_model_data,
    duration_col="tenure_days",
    event_col="churned"
)

C:\Users\ManishPanda\anaconda3\Lib\site-packages\lifelines\utils\__init__.py:1163: ConvergenceWarning: Column engagement_score has high sample correlation with the duration column. This may harm convergence. This could be a form of 'complete separation'.     See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression

  warnings.warn(dedent(warning_text), ConvergenceWarning)
C:\Users\ManishPanda\anaconda3\Lib\site-packages\lifelines\utils\__init__.py:1163: ConvergenceWarning: Column usage_trend has high sample correlation with the duration column. This may harm convergence. This could be a form of 'complete separation'.     See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression

  warnings.warn(dedent(warning_text), ConvergenceWarning)
C:\Users\ManishPanda\anaconda3\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1614: ConvergenceWarning: Newton-Raphson failed to converge suff

<lifelines.CoxPHFitter: fitted with 6 total observations, 3 right-censored observations>

In [11]:
from lifelines import CoxPHFitter

cph = CoxPHFitter(penalizer=0.1)

In [12]:
cph.fit(
    survival_model_data,
    duration_col="tenure_days",
    event_col="churned"
)

<lifelines.CoxPHFitter: fitted with 6 total observations, 3 right-censored observations>

In [13]:
cph.print_summary()

<lifelines.CoxPHFitter: fitted with 6 total observations, 3 right-censored observations>
             duration col = 'tenure_days'
                event col = 'churned'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 6
number of events observed = 3
   partial log-likelihood = -1.83
         time fit was run = 2026-01-07 16:37:48 UTC

---
                  coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                         
engagement_score -0.03      0.97      0.04           -0.11            0.04                0.90                1.04
usage_trend      -4.38      0.01      4.26          -12.73            3.96                0.00               52.39

                  cmp to     z    p  -log2(p)
covariate                                    
engagement_score    0.00 -0.95 0.34      1.55
usage_trend         0.00 -1.03 0.30      1.72
---
Concordance = 1.00
Partial AIC = 7.65
log-likelihood ratio test = 5.92 on 2 df
-log2(p) of ll-ratio test = 4.27

In [14]:
cph.predict_survival_function(survival_data.iloc[[1]])

,1
90.0,0.884575
120.0,0.658097
180.0,0.285339
300.0,0.285339
400.0,0.285339
500.0,0.285339
